<a href="https://colab.research.google.com/github/ynam0327-afk/REDRED/blob/main/Cross_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**긴급재난문자 공식 API 연동**

In [1]:
import re
import difflib
import time
import requests
import urllib3
import pandas as pd

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

API_URL = "https://www.safetydata.go.kr/V2/api/DSSP-IF-00247"
SERVICE_KEY = "긴급재난문자"

REQUEST_TIMEOUT_SEC = 15
MAX_RETRIES = 3

EXACT_MATCH_THRESHOLD = 0.9


def fetch_official_sms(crt_dt: str = None, rgn_nm: str = None,
                        num_of_rows: int = 100, page_no: int = 1) -> list:

    params = {
        "serviceKey": SERVICE_KEY,
        "returnType": "json",
        "numOfRows": num_of_rows,
        "pageNo": page_no,
    }
    if crt_dt:
        params["crtDt"] = crt_dt
    if rgn_nm:
        params["rgnNm"] = rgn_nm

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(API_URL, params=params, timeout=REQUEST_TIMEOUT_SEC, verify=False)
            resp.raise_for_status()
            data = resp.json()
            # 실제 응답 확인됨: 최상위 키는 'body'
            items = data.get("body", [])
            return items
        except requests.exceptions.RequestException as e:
            print(f"[요청 실패 {attempt}/{MAX_RETRIES}] {e}")
            if attempt == MAX_RETRIES:
                raise
            time.sleep(2 * attempt)


def normalize_for_compare(text: str) -> str:
    """비교 전 공백/개행 등을 정규화 - 줄바꿈 위치 차이 정도로 유사도가 떨어지지 않게."""
    if not isinstance(text, str):
        return ""
    return re.sub(r"\s+", " ", text).strip()


def find_official_match(pasted_text: str, records: list) -> dict:
    """
    사용자가 붙여넣은 텍스트와 가장 유사한 공식 발송 기록을 찾는다.
    임계값(0.9) 이상이어야 "진짜 발송분과 일치"로 인정한다.
    """
    if not records:
        return {"matched": False, "score": 0.0, "record": None}

    norm_pasted = normalize_for_compare(pasted_text)
    best_score, best_record = 0.0, None

    for rec in records:
        msg = normalize_for_compare(rec.get("MSG_CN", ""))
        score = difflib.SequenceMatcher(None, norm_pasted, msg).ratio()
        if score > best_score:
            best_score, best_record = score, rec

    return {
        "matched": best_score >= EXACT_MATCH_THRESHOLD,
        "score": round(best_score, 4),
        "record": best_record,
    }


def official_sms_reliability(pasted_text: str, crt_dt: str, rgn_nm: str = None) -> dict:
    """
    사용자가 붙여넣은 문자를 공식 API 발송 이력과 대조해 신뢰도를 낸다.
    cross_validation.py의 combined_disaster_reliability보다 우선 적용할
    최상위 신호로 설계 - 여기서 매칭되면 사실상 진위가 확정된다.
    """
    try:
        records = fetch_official_sms(crt_dt=crt_dt, rgn_nm=rgn_nm, num_of_rows=100)
    except Exception as e:
        return {"score": None, "note": f"API 조회 실패 - {e} (하위 소스로 폴백 필요)"}

    result = find_official_match(pasted_text, records)

    if result["matched"]:
        return {
            "score": 1.0,
            "note": f"공식 발송 이력과 일치(유사도 {result['score']}) - 진위 확정",
            "matched_record": result["record"],
        }

    return {
        "score": None,  # 확정 불가 - 다른 소스로 폴백해야 함 (0.5로 단정하지 않음)
        "note": f"공식 발송 이력에서 일치하는 문자를 못 찾음(최고 유사도 {result['score']}) - 하위 소스로 폴백 필요",
    }


# ---------------------------------------------------------------------------
# 실행 예시
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    # 1) 먼저 실제 응답 구조부터 확인 (최상위 키 이름, 필드 실제 값 형태)
    sample = fetch_official_sms(num_of_rows=3)
    print("응답 샘플:", sample)


응답 샘플: [{'MSG_CN': '[경기남부경찰청] 김포시에서 배회중인 김학균씨(남,40세)를 찾습니다-179cm,75kg,녹색체크무늬셔츠,베이지색 하의와 신발 vo.la/Wv53w / ☎182', 'RCPTN_RGN_NM': '경기도 김포시 ', 'CRT_DT': '2023/09/19 12:22:17', 'REG_YMD': '2023-09-19', 'EMRG_STEP_NM': '안전안내', 'SN': 205355, 'DST_SE_NM': '기타', 'MDFCN_YMD': '2023-09-19'}, {'MSG_CN': '[서울경찰청] 노원구에서 실종된 김현주씨(여,41세)를 찾습니다 -160cm,68kg,꽃무늬티,회색칠부바지,검정슬리퍼\r\nvo.la/bhWwm / ☎182', 'RCPTN_RGN_NM': '서울특별시 노원구 ', 'CRT_DT': '2023/09/19 13:30:45', 'REG_YMD': '2023-09-19', 'EMRG_STEP_NM': '안전안내', 'SN': 205356, 'DST_SE_NM': '기타', 'MDFCN_YMD': '2023-09-19'}, {'MSG_CN': '[서울경찰청] 용산구에서 배회중인 신철화씨(남,62세)를 찾습니다-165cm,80kg,흰색환자복상하의,흰색운동화,대머리 vo.la/8giBG / ☎182', 'RCPTN_RGN_NM': '서울특별시 용산구 ', 'CRT_DT': '2023/09/19 13:36:14', 'REG_YMD': '2023-09-19', 'EMRG_STEP_NM': '안전안내', 'SN': 205357, 'DST_SE_NM': '기타', 'MDFCN_YMD': '2023-09-19'}]


/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.safetydata.go.kr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


**Cross Validation**

In [26]:
from dataclasses import dataclass
from typing import Optional

!jupyter nbconvert --to script /content/REDRED/content_authenticity.ipynb

import os
os.rename("/content/REDRED/content_authenticity.txt", "/content/REDRED/content_authenticity.py")

import sys
sys.path.insert(0, "/content/REDRED")

# ---------------------------------------------------------------------------
# 1. 재난정보 교차검증
# ---------------------------------------------------------------------------

@dataclass
class SourceMatch:
    """개별 소스(소방청DB 또는 119신고접수)의 매칭 결과를 표준화한 형태."""
    matched: bool
    confidence: float = 0.0   # 그 소스 안에서의 신뢰도(예: text_similarity, 시각근접도 환산값)
    note: Optional[str] = None


from content_authenticity import content_authenticity_score


def combined_disaster_reliability(fire_db: Optional[SourceMatch],
                                   call119: Optional[SourceMatch],
                                   message: Optional[str] = None,
                                   official_match_score: Optional[float] = None) -> dict:

    if official_match_score is not None:
        return {
            "score": official_match_score,
            "matched_sources": ["official_sms_api"],
            "note": "긴급재난문자 공식 API 발송 이력과 원문 일치 - 진위 확정(최우선 신호)",
        }

    available = [s for s in (fire_db, call119) if s is not None]

    if not available:
        if message:
            content = content_authenticity_score(message)
            return {
                "score": content["score"],
                "matched_sources": [],
                "note": f"DB 조회 불가 - 콘텐츠 기반 판단으로 대체: {content['note']}",
            }
        return {
            "score": 0.5,
            "matched_sources": [],
            "note": "정보 부족 - 조회 가능한 소스가 아예 없음(판단보류, 위험 아님)",
        }

    fire_matched = fire_db is not None and fire_db.matched
    call119_matched = call119 is not None and call119.matched

    if not fire_matched and not call119_matched:
        if message:
            content = content_authenticity_score(message)
            return {
                "score": content["score"],
                "matched_sources": [],
                "note": f"DB 조회했지만 대응 사건 없음 - 콘텐츠 기반 판단으로 대체: {content['note']}",
            }
        return {
            "score": 0.5,
            "matched_sources": [],
            "note": "정보 부족 - 조회했지만 대응 사건 없음(판단보류, 위험 아님)",
        }

    if fire_matched and call119_matched:
        avg_conf = (fire_db.confidence + call119.confidence) / 2
        score = 0.85 + 0.15 * avg_conf
        note = "두 소스(fire_db, call119) 모두 매칭 - 최고 신뢰도"
        matched_sources = ["fire_db", "call119"]
    elif call119_matched:
        score = 0.75 + 0.25 * call119.confidence
        note = "단일 소스(call119)만 매칭 - 실증적으로 신뢰도 높은 주 신호"
        matched_sources = ["call119"]
    else:
        score = 0.6 + 0.2 * fire_db.confidence
        note = "단일 소스(fire_db)만 매칭 - 실전에서 드물게만 기여하는 보조 신호"
        matched_sources = ["fire_db"]

    return {
        "score": round(min(score, 1.0), 3),
        "matched_sources": matched_sources,
        "note": note,
    }


# ---------------------------------------------------------------------------
# 2. URL 교차검증 (화이트리스트 + RF 모델)
# ---------------------------------------------------------------------------

def url_cross_check(is_whitelisted: bool, rf_score: float) -> dict:
    """
    화이트리스트와 RF 모델 결과를 합쳐 URL 위험도와 판정 사유를 낸다.
    외부 보안 API는 이번 주 범위에서 제외(선택 과제로 보류)했으므로
    두 신호만 사용한다.
    """
    if is_whitelisted:
        return {"risk": 0.0, "reason": "화이트리스트 매칭 - 공식 도메인"}

    if rf_score >= 0.7:
        return {"risk": rf_score, "reason": "RF 모델 고위험 판정"}
    if rf_score <= 0.3:
        return {"risk": rf_score, "reason": "RF 모델 저위험 판정"}

    return {"risk": rf_score, "reason": "RF 모델 애매 구간 - review 대상"}


# ---------------------------------------------------------------------------
# 3. 최종 통합 스코어
# ---------------------------------------------------------------------------

def final_alert_score_v2(fire_db: Optional[SourceMatch], call119: Optional[SourceMatch],
                          is_whitelisted: bool, rf_score: float,
                          message: Optional[str] = None,
                          official_match_score: Optional[float] = None,
                          w1: float = 0.6, w2: float = 0.4) -> dict:
    disaster = combined_disaster_reliability(fire_db, call119, message, official_match_score)
    url = url_cross_check(is_whitelisted, rf_score)

    final = w1 * disaster["score"] + w2 * (1 - url["risk"])

    return {
        "disaster_reliability_score": disaster["score"],
        "disaster_matched_sources": disaster["matched_sources"],
        "disaster_note": disaster["note"],
        "url_risk_score": url["risk"],
        "url_reason": url["reason"],
        "final_score": round(final, 3),
        "decision": "approve" if final >= 0.7 else ("review" if final >= 0.4 else "reject"),
    }


# ---------------------------------------------------------------------------
# 검증 - 확장 시나리오
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    scenarios = [
        ("두 소스 모두 매칭 + 안전 URL",
         SourceMatch(True, 0.8), SourceMatch(True, 0.9), True, 0.0),
        ("소방청DB만 매칭 + 안전 URL",
         SourceMatch(True, 0.6), None, True, 0.0),
        ("119만 매칭 + 안전 URL",
         SourceMatch(False), SourceMatch(True, 0.9), True, 0.0),
        ("서울/부산 외 지역, 소방청 매칭 없음 (119 커버리지 밖)",
         SourceMatch(False), None, False, 0.1),
        ("정보부족 + RF 애매",
         SourceMatch(False), SourceMatch(False), False, 0.5),
        ("정보부족 + 위험 URL",
         SourceMatch(False), SourceMatch(False), False, 0.9),
        ("두 소스 불일치(소방청은 매칭, 119는 매칭 안됨) + 안전 URL",
         SourceMatch(True, 0.5), SourceMatch(False), True, 0.0),
    ]

    for name, fire_db, call119, wl, rf in scenarios:
        result = final_alert_score_v2(fire_db, call119, wl, rf)
        print(f"[{name}]")
        print(f"  {result}\n")

    print("=== 콘텐츠 기반 판단 (DB 매칭 전혀 없을 때 원문으로 판단) ===")
    content_scenarios = [
        ("DB 매칭 없음 + 정상적인 공식 문구 + 안전 URL",
         "오늘 09:34 원주시 원동 한주아파트 101동 건물에서 화재 발생. 차량은 건물 주변 도로를 우회하고, 건물 내 시민은 건물 밖으로 대피하세요. [원주시]",
         True, 0.0),
        ("DB 매칭 없음 + 스미싱 의심 문구",
         "재난지원금 무료 쿠폰 당첨! 즉시 확인 후 인증번호 입력 http://bit.ly/abc123",
         False, 0.6),
        ("DB 매칭 없음 + 발신기관 태그 자체가 없음",
         "화재가 발생했습니다 주의하세요",
         True, 0.0),
    ]
    for name, message, wl, rf in content_scenarios:
        result = final_alert_score_v2(SourceMatch(False), SourceMatch(False), wl, rf, message=message)
        print(f"[{name}]")
        print(f"  {result}\n")

    print("=== 공식 재난문자 API 일치 (최우선 신호) ===")
    official_scenarios = [
        ("공식 API와 원문 일치 + 안전 URL",
         1.0, True, 0.0),
        ("공식 API 미일치 -> 기존 로직 폴백 + 안전 URL",
         None, True, 0.0),
    ]
    for name, official_score, wl, rf in official_scenarios:
        result = final_alert_score_v2(SourceMatch(False), SourceMatch(False), wl, rf,
                                       official_match_score=official_score)
        print(f"[{name}]")
        print(f"  {result}\n")


[NbConvertApp] Converting notebook /content/REDRED/content_authenticity.ipynb to script
[NbConvertApp] Writing 3478 bytes to /content/REDRED/content_authenticity.txt
[두 소스 모두 매칭 + 안전 URL]
  {'disaster_reliability_score': 0.978, 'disaster_matched_sources': ['fire_db', 'call119'], 'disaster_note': '두 소스(fire_db, call119) 모두 매칭 - 최고 신뢰도', 'url_risk_score': 0.0, 'url_reason': '화이트리스트 매칭 - 공식 도메인', 'final_score': 0.987, 'decision': 'approve'}

[소방청DB만 매칭 + 안전 URL]
  {'disaster_reliability_score': 0.72, 'disaster_matched_sources': ['fire_db'], 'disaster_note': '단일 소스(fire_db)만 매칭 - 실전에서 드물게만 기여하는 보조 신호', 'url_risk_score': 0.0, 'url_reason': '화이트리스트 매칭 - 공식 도메인', 'final_score': 0.832, 'decision': 'approve'}

[119만 매칭 + 안전 URL]
  {'disaster_reliability_score': 0.975, 'disaster_matched_sources': ['call119'], 'disaster_note': '단일 소스(call119)만 매칭 - 실증적으로 신뢰도 높은 주 신호', 'url_risk_score': 0.0, 'url_reason': '화이트리스트 매칭 - 공식 도메인', 'final_score': 0.985, 'decision': 'approve'}

[서울/부산 외 지역, 소방청 매칭 없음 (1